[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_11/2_sql.ipynb)

# Day 11: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (29 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Best artists inside the biggest genres | medium | 6 |
| Q2 | Runner-up customer in every country | medium | 6 |
| Q3 | Top spenders per country in March | medium | 5 |
| Q4 | Best movies of each decade, fairly | medium | 7 |
| Q5 | Every employee on the sales board (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: MovieLens small (GroupLens, research and education use).
def movielens(short):
    path = os.path.join(DATA, f"movielens_{short}.csv")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading MovieLens")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read()))
        for s in ("ratings", "movies", "tags", "links"):
            with open(os.path.join(DATA, f"movielens_{s}.csv"), "wb") as f:
                f.write(z.read(f"ml-latest-small/{s}.csv"))
    return pd.read_csv(path)

_r = movielens("ratings")
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]] \
    .to_sql("ml_ratings", db, index=False)
movielens("movies").rename(columns={"movieId": "movie_id"}).to_sql("ml_movies", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title` (with the year), `genres` (several genres joined by a vertical bar, e.g. Comedy\|Romance) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Best artists inside the biggest genres

*medium, about 6 min*

The music store wants a slide: for each of its **4 genres with the most revenue**, the **3 artists**
with the most revenue in that genre. Revenue of an invoice line is `UnitPrice * Quantity`.

If two artists tie for third place, show both (a slide that hides one of two equal artists is wrong).

Return `genre`, `artist`, `revenue` (rounded to 2 decimals), `rnk`, ordered by genre, rank, artist.
Example of the shape: `Rock | U2 | 90.09 | 1`.

Follow-up to answer out loud: what changes if the business says "exactly 3 rows per genre, break ties by artist
name"?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "top K inside each group". Pattern: rank inside a window `PARTITION BY genre ORDER BY revenue DESC`, then filter rank <= 3 in an outer query. Ties kept means RANK (or DENSE_RANK), not ROW_NUMBER.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE 1: genre revenue, keep the top 4 with ORDER BY ... LIMIT 4.
2. CTE 2: revenue per (genre, artist) for those genres: InvoiceLine, Track, Genre, Album, Artist.
3. CTE 3: `RANK() OVER (PARTITION BY genre ORDER BY revenue DESC)`.
4. Outer query: `WHERE rnk <= 3`. You cannot filter a window function in WHERE of the same SELECT.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH genre_rev AS (
    SELECT g.Name AS genre, SUM(il.UnitPrice * il.Quantity) AS rev
    FROM InvoiceLine il
    JOIN Track t ON t.TrackId = il.TrackId
    JOIN Genre g ON g.GenreId = t.GenreId
    GROUP BY g.Name
    ORDER BY rev DESC
    LIMIT 4
), artist_rev AS (
    SELECT g.Name AS genre, ar.Name AS artist, ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS revenue
    FROM InvoiceLine il
    JOIN Track t   ON t.TrackId = il.TrackId
    JOIN Genre g   ON g.GenreId = t.GenreId
    JOIN Album al  ON al.AlbumId = t.AlbumId
    JOIN Artist ar ON ar.ArtistId = al.ArtistId
    WHERE g.Name IN (SELECT genre FROM genre_rev)
    GROUP BY g.Name, ar.Name
), ranked AS (
    SELECT genre, artist, revenue,
           RANK() OVER (PARTITION BY genre ORDER BY revenue DESC) AS rnk
    FROM artist_rev
)
SELECT genre, artist, revenue, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY genre, rnk, artist
```

Result:

| genre | artist | revenue | rnk |
|---|---|---|---|
| Alternative & Punk | Titãs | 33.66 | 1 |
| Alternative & Punk | Faith No More | 32.67 | 2 |
| Alternative & Punk | Green Day | 32.67 | 2 |
| Latin | Os Paralamas Do Sucesso | 44.55 | 1 |
| Latin | Chico Buarque | 26.73 | 2 |
| Latin | Chico Science & Nação Zumbi | 24.75 | 3 |
| Metal | Metallica | 90.09 | 1 |
| Metal | Iron Maiden | 69.3 | 2 |
| Metal | Bruce Dickinson | 11.88 | 3 |
| Metal | System Of A Down | 11.88 | 3 |
| Rock | U2 | 90.09 | 1 |
| Rock | Led Zeppelin | 86.13 | 2 |

(13 rows, first 12 shown)

**Why:** The window ranks artists separately inside each genre. RANK gives tied artists the same number, so Faith No More and Green Day both get 2 in Alternative & Punk, and two Metal artists share rank 3, which is why Metal has 4 rows. With RANK the next rank after a tie is skipped (1, 2, 2, 4); DENSE_RANK would not skip (1, 2, 2, 3) and could return even more rows. For "exactly 3 rows" use `ROW_NUMBER() OVER (PARTITION BY genre ORDER BY revenue DESC, artist)`: the extra sort key makes the result deterministic.

**Complexity:** One pass over InvoiceLine for the joins and group by, then a sort per genre for the window.

**Common mistakes:** Filtering `WHERE RANK() OVER (...) <= 3` (not allowed: windows are computed after WHERE). Using ROW_NUMBER without a tie breaker, so the result changes between runs. Ranking over the whole table and forgetting PARTITION BY. Grouping by artist only, which merges an artist across genres.

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking), [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte)

</details>

---
### Q2. Runner-up customer in every country

*medium, about 6 min*

For **every** country in `Customer`, find the customer with the **second highest total spend**
(sum of their invoice totals). Return `country`, `second_customer`, `second_spend` (2 decimals).

Rules:
- A country with only one customer must still appear, with NULL in the other two columns.
- "Second highest" means the second highest **distinct** amount. If three customers spent 40, 40 and 37, the
  runner-up is the one with 37. If several customers share the second amount, show all of them.

Example: in a country with spends 43.62, 43.62, 43.62 (all equal) there is no second amount, so you return NULL.

This is the per-group version of the classic "second highest salary" question.

*Source: [LeetCode 176 (Second Highest Salary), same idea](https://leetcode.com/problems/second-highest-salary/)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "Nth highest, and show NULL when it does not exist". Pattern: DENSE_RANK per country (distinct amounts) plus a LEFT JOIN from the full list of countries.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: spend per customer (Customer join Invoice, group by customer).
2. CTE: `DENSE_RANK() OVER (PARTITION BY country ORDER BY spend DESC) AS dr`.
3. Start from `SELECT DISTINCT Country FROM Customer` and LEFT JOIN the ranked rows `ON country matches AND dr = 2`. Put `dr = 2` in the ON clause, not in WHERE.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH spend AS (
    SELECT c.Country AS country, c.FirstName || ' ' || c.LastName AS customer, SUM(i.Total) AS spend
    FROM Customer c
    JOIN Invoice i ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, c.Country, c.FirstName, c.LastName
), ranked AS (
    SELECT country, customer, spend,
           DENSE_RANK() OVER (PARTITION BY country ORDER BY spend DESC) AS dr
    FROM spend
)
SELECT c.country, r.customer AS second_customer, ROUND(r.spend, 2) AS second_spend
FROM (SELECT DISTINCT Country AS country FROM Customer) AS c
LEFT JOIN ranked AS r
       ON r.country = c.country AND r.dr = 2
ORDER BY c.country
```

Result:

| country | second_customer | second_spend |
|---|---|---|
| Argentina | NULL | NULL |
| Australia | NULL | NULL |
| Austria | NULL | NULL |
| Belgium | NULL | NULL |
| Brazil | Alexandre Rocha | 37.62 |
| Brazil | Eduardo Martins | 37.62 |
| Brazil | Fernanda Ramos | 37.62 |
| Brazil | Roberto Almeida | 37.62 |
| Canada | Jennifer Peterson | 38.62 |
| Chile | NULL | NULL |
| Czech Republic | František Wichterlová | 40.62 |
| Denmark | NULL | NULL |

(30 rows, first 12 shown)

**Why:** DENSE_RANK numbers distinct amounts 1, 2, 3 without gaps, so `dr = 2` is exactly "the second distinct amount" and returns every customer who has it (Brazil shows 4 tied customers). Countries with one customer, or where everybody spent the same (the United Kingdom has 3 customers with equal spend), have no `dr = 2` row, and the LEFT JOIN keeps them with NULL. With RANK, ties at the top would push the next amount to rank 3 and you would wrongly get NULL; with ROW_NUMBER you would return a tied first-place customer as "second".

**Complexity:** Group by over invoices, one sorted window per country.

**Common mistakes:** Writing `WHERE r.dr = 2` after the LEFT JOIN: it removes the NULL rows and turns the LEFT JOIN into an inner join. Using `LIMIT 1 OFFSET 1`, which only works for one group and ignores ties. Using `Invoice.BillingCountry` instead of the customer's country (they can differ in real data).

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q3. Top spenders per country in March

*medium, about 5 min*

Using the made-up shopping app: for each `country`, return the **2 users** with the highest
**completed** purchase amount in **March 2024** (`purchases.status = 'completed'`, refunds do not count).
Exactly 2 rows per country: if two users have the same spend, the one whose first March order came earlier wins.

Return `country`, `rn` (1 or 2), `user_id`, `spend` (2 decimals), ordered by country and rn.

Follow-ups: (a) Why is the date filter written as `order_time >= '2024-03-01' AND order_time < '2024-04-01'`
and not `BETWEEN '2024-03-01' AND '2024-03-31'`? (b) What would you change for "top 2 per country per week"?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "top 2 per country, exactly 2 rows". Pattern: ROW_NUMBER over a partition with an explicit tie breaker, then filter rn <= 2.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: join purchases to app_users, filter completed and March, group by country and user: `SUM(amount)` and `MIN(order_time)`.
2. `ROW_NUMBER() OVER (PARTITION BY country ORDER BY spend DESC, first_order)`.
3. Keep rn <= 2.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH march AS (
    SELECT u.country, p.user_id, SUM(p.amount) AS spend, MIN(p.order_time) AS first_order
    FROM purchases p
    JOIN app_users u ON u.user_id = p.user_id
    WHERE p.status = 'completed'
      AND p.order_time >= '2024-03-01' AND p.order_time < '2024-04-01'
    GROUP BY u.country, p.user_id
), ranked AS (
    SELECT country, user_id, ROUND(spend, 2) AS spend,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY spend DESC, first_order) AS rn
    FROM march
)
SELECT country, rn, user_id, spend
FROM ranked
WHERE rn <= 2
ORDER BY country, rn
```

Result:

| country | rn | user_id | spend |
|---|---|---|---|
| BR | 1 | 1485 | 232.75 |
| BR | 2 | 908 | 228.51 |
| DE | 1 | 681 | 186.23 |
| DE | 2 | 384 | 154.75 |
| ID | 1 | 390 | 284.4 |
| ID | 2 | 537 | 158.93 |
| KR | 1 | 963 | 268.38 |
| KR | 2 | 153 | 253.94 |
| US | 1 | 1464 | 297.68 |
| US | 2 | 945 | 212.63 |

**Why:** ROW_NUMBER always gives 1, 2, 3 with no ties, so each country returns exactly 2 rows; the second sort key (first order time) decides ties in a way the business can explain. (a) `order_time` has a time part: `BETWEEN '2024-03-01' AND '2024-03-31'` compares text and drops every order on March 31 after 00:00:00. A half-open range `>= start AND < next start` is always correct, also in PostgreSQL with timestamps. (b) Add the week to the GROUP BY and to the PARTITION BY: `PARTITION BY country, week`.

**Complexity:** Filter and group by on purchases, then one window sort per country.

**Common mistakes:** Ranking before filtering March (ranks lifetime spend). Forgetting the status filter. Partitioning by user instead of country. Using RANK when the spec says exactly 2 rows.

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q4. Best movies of each decade, fairly

*medium, about 7 min*

MovieLens (real ratings). The year of a movie is in its title, e.g. `Toy Story (1995)`.
For each **decade from 1970 on**, return the **3 movies with the highest average rating**, counting only movies with
**at least 50 ratings** (otherwise a movie rated once with 5 stars wins). Ties on the average: more ratings first.

Return `decade` (1970, 1980, ...), `rn`, `title`, `avg_rating` (2 decimals), `n_ratings`.

Hints on the data: a few titles have no year or trailing spaces; skip titles that do not end with `(YYYY)` after
trimming. In SQLite, `substr(s, -5, 4)` takes 4 characters starting 5 from the end.

Follow-up: the product manager does not like a hard cutoff of 50. What would you propose instead?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "top 3 per decade" plus "only if enough data". Pattern: aggregate per movie with HAVING (minimum count), then ROW_NUMBER per decade.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE per movie: decade = `CAST(substr(trim(title), -5, 4) AS INTEGER) / 10 * 10`, COUNT(*), AVG(rating); WHERE trimmed title LIKE '%(____)'; HAVING COUNT(*) >= 50.
2. `ROW_NUMBER() OVER (PARTITION BY decade ORDER BY avg_rating DESC, n_ratings DESC)`.
3. Keep rn <= 3 and decade >= 1970.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH movie_stats AS (
    SELECT m.movie_id, m.title,
           CAST(substr(trim(m.title), -5, 4) AS INTEGER) / 10 * 10 AS decade,
           COUNT(*) AS n_ratings,
           AVG(r.rating) AS avg_rating
    FROM ml_movies m
    JOIN ml_ratings r ON r.movie_id = m.movie_id
    WHERE trim(m.title) LIKE '%(____)'
    GROUP BY m.movie_id, m.title
    HAVING COUNT(*) >= 50
), ranked AS (
    SELECT decade, title, n_ratings, ROUND(avg_rating, 2) AS avg_rating,
           ROW_NUMBER() OVER (PARTITION BY decade ORDER BY avg_rating DESC, n_ratings DESC) AS rn
    FROM movie_stats
)
SELECT decade, rn, title, avg_rating, n_ratings
FROM ranked
WHERE rn <= 3 AND decade >= 1970
ORDER BY decade, rn
```

Result:

| decade | rn | title | avg_rating | n_ratings |
|---|---|---|---|---|
| 1970 | 1 | Godfather, The (1972) | 4.29 | 192 |
| 1970 | 2 | Godfather: Part II, The (1974) | 4.26 | 129 |
| 1970 | 3 | Star Wars: Episode IV - A New Hope (1977) | 4.23 | 251 |
| 1980 | 1 | Princess Bride, The (1987) | 4.23 | 142 |
| 1980 | 2 | Star Wars: Episode V - The Empire Strikes Back (1980) | 4.22 | 211 |
| 1980 | 3 | Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) | 4.21 | 200 |
| 1990 | 1 | Shawshank Redemption, The (1994) | 4.43 | 317 |
| 1990 | 2 | Fight Club (1999) | 4.27 | 218 |
| 1990 | 3 | Goodfellas (1990) | 4.25 | 126 |
| 2000 | 1 | Departed, The (2006) | 4.25 | 107 |
| 2000 | 2 | Dark Knight, The (2008) | 4.24 | 149 |
| 2000 | 3 | Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001) | 4.18 | 120 |

(15 rows, first 12 shown)

**Why:** HAVING filters groups after aggregation, so the minimum count is applied per movie before ranking. Integer division `year / 10 * 10` turns 1994 into 1990. The result is sensible: The Godfather leads the 1970s and The Shawshank Redemption the 1990s. Follow-up: a hard cutoff throws away information. A Bayesian (shrunk) average, `(n * avg + m * global_avg) / (n + m)` with m around 30 to 50, pulls movies with few ratings toward the global mean, so they can rank but need more evidence to reach the top. Dialects: PostgreSQL has `substring(title from '\((\d{4})\)\s*$')` for a regex extract, and integer division also truncates there.

**Complexity:** One group by over 100k ratings, then a small sort per decade.

**Common mistakes:** Putting the count condition in WHERE (aggregates are not allowed there). Using AVG on an integer column in PostgreSQL is fine, but `SUM(x) / COUNT(*)` on integers truncates. Taking the first 4 digits in the title instead of the last ones (titles like '1984 (1984)').

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q5. Every employee on the sales board (review)

*medium, about 5 min*

Make a leaderboard of **all 8 employees**: `EmployeeId`, `employee` (first and last name),
`Title`, `customers` (customers they support), `revenue` (sum of those customers' invoice totals, 2 decimals) and
`revenue_rank` (1 = most revenue, ties share a rank and the next rank does not skip).

Employees who support no customers must appear with 0 customers and 0.00 revenue (not NULL). Example row:
`1 | Andrew Adams | General Manager | 0 | 0.0 | 4`.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "all employees, even without customers" plus a ranking without gaps. Pattern: LEFT JOIN chain with COALESCE, then DENSE_RANK over the aggregated value.

</details>

<details><summary><b>Hint 2</b></summary>

1. Employee LEFT JOIN Customer ON SupportRepId, LEFT JOIN Invoice ON CustomerId.
2. Group by employee; `COUNT(DISTINCT c.CustomerId)` and `COALESCE(SUM(i.Total), 0)`.
3. `DENSE_RANK() OVER (ORDER BY COALESCE(SUM(i.Total), 0) DESC)`: a window can use an aggregate in the same SELECT, because windows run after GROUP BY.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT e.EmployeeId,
       e.FirstName || ' ' || e.LastName AS employee,
       e.Title,
       COUNT(DISTINCT c.CustomerId)          AS customers,
       ROUND(COALESCE(SUM(i.Total), 0), 2)   AS revenue,
       DENSE_RANK() OVER (ORDER BY COALESCE(SUM(i.Total), 0) DESC) AS revenue_rank
FROM Employee e
LEFT JOIN Customer c ON c.SupportRepId = e.EmployeeId
LEFT JOIN Invoice  i ON i.CustomerId = c.CustomerId
GROUP BY e.EmployeeId, e.FirstName, e.LastName, e.Title
ORDER BY revenue_rank, e.EmployeeId
```

Result:

| EmployeeId | employee | Title | customers | revenue | revenue_rank |
|---|---|---|---|---|---|
| 3 | Jane Peacock | Sales Support Agent | 21 | 833.04 | 1 |
| 4 | Margaret Park | Sales Support Agent | 20 | 775.4 | 2 |
| 5 | Steve Johnson | Sales Support Agent | 18 | 720.16 | 3 |
| 1 | Andrew Adams | General Manager | 0 | 0.0 | 4 |
| 2 | Nancy Edwards | Sales Manager | 0 | 0.0 | 4 |
| 6 | Michael Mitchell | IT Manager | 0 | 0.0 | 4 |
| 7 | Robert King | IT Staff | 0 | 0.0 | 4 |
| 8 | Laura Callahan | IT Staff | 0 | 0.0 | 4 |

**Why:** Only the 3 sales support agents have customers; the 5 others survive because every join is a LEFT JOIN. `COUNT(DISTINCT c.CustomerId)` counts customers, not invoices (the invoice join repeats each customer once per invoice), and COUNT ignores NULLs so it gives 0. SUM over no rows is NULL, hence COALESCE. DENSE_RANK gives the 5 employees with 0 revenue rank 4, right after rank 3.

**Complexity:** One pass over the joined rows (8 employees, 59 customers, 412 invoices).

**Common mistakes:** `COUNT(*)` (gives 1 for an employee with no customers, because the LEFT JOIN row exists). `COUNT(c.CustomerId)` without DISTINCT (counts invoices). An inner JOIN to Invoice after the LEFT JOIN to Customer, which drops the employees again.

**Learn more:** [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_11/3_stats.ipynb)